In [1]:
import math
import numpy as np
import wandb
import pickle
import os
import shapely.wkt as wkt
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
from torch_geometric.transforms import LineGraph

import gzip
import xml.etree.ElementTree as ET

import torch
import torch_geometric
from torch_geometric.data import Data

import processing_io as pio
import sys
import os
import joblib
import json

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from shapely.geometry import Point, LineString, box
from matplotlib.colors import TwoSlopeNorm

from shapely.ops import unary_union
from mpl_toolkits.axes_grid1 import make_axes_locatable
from torch_geometric.data import Data, Batch
import torch
from torch_geometric.data import Data
import help_functions as hf

districts = gpd.read_file("../../data/visualisation/districts_paris.geojson")

# Add the 'scripts' directory to the Python path
scripts_path = os.path.abspath(os.path.join('..'))
if scripts_path not in sys.path:
    sys.path.append(scripts_path)

import gnn_io as gio
import gnn_architectures as garch

highway_mapping = {
    'trunk': 0, 'trunk_link': 0, 'motorway_link': 0,
    'primary': 1, 'primary_link': 1,
    'secondary': 2, 'secondary_link': 2,
    'tertiary': 3, 'tertiary_link': 3,
    'residential': 4, 'living_street': 5,
    'pedestrian': 6, 'service': 7,
    'construction': 8, 'unclassified': 9,
    'np.nan': -1
}

def compute_r2_torch_with_mean_targets(mean_targets, preds, targets):
    ss_tot = torch.sum((targets - mean_targets) ** 2)
    ss_res = torch.sum((targets - preds) ** 2)
    r2 = 1 - (ss_res / ss_tot)
    return r2

def validate_one_model(model, data, loss_func, device):
    model.eval()
    pred = []
    actual = []
    with torch.inference_mode():
        input_node_features, targets = data.x.to(device), data.y.to(device)
        predicted = model(data.to(device))
        # print(predicted.shape)
        pred.append(predicted)
        actual.append(targets)
        val_loss = loss_func(predicted, targets).item()
    actual_vals = torch.cat(actual)
    predicted_vals = torch.cat(pred)
    
    mean_targets = torch.mean(targets)
    r_squared = compute_r2_torch_with_mean_targets(mean_targets = mean_targets, preds=predicted_vals, targets=actual_vals)
    baseline_loss = loss_func(targets, torch.full_like(predicted_vals, mean_targets))
    return val_loss, r_squared, targets, predicted, baseline_loss


import math
import numpy as np
import wandb
import pickle
import os
import shapely.wkt as wkt
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
from torch_geometric.transforms import LineGraph

import gzip
import xml.etree.ElementTree as ET

import torch
import torch_geometric
from torch_geometric.data import Data

import processing_io as pio
import sys
import os
import joblib
import json

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from shapely.geometry import Point, LineString, box
from matplotlib.colors import TwoSlopeNorm

from shapely.ops import unary_union
from mpl_toolkits.axes_grid1 import make_axes_locatable
from torch_geometric.data import Data, Batch
import torch
from torch_geometric.data import Data
import alphashape
from matplotlib.lines import Line2D
from shapely.geometry import Polygon

def plot_combined_output(gdf_input: gpd.GeoDataFrame, column_to_plot: str, font: str = 'Times New Roman', 
                         save_it: bool = False, number_to_plot: int = 0,
                         zone_to_plot:str= "this_zone",
                         is_predicted: bool = False, alpha:int=100, 
                         use_fixed_norm:bool=True, 
                         fixed_norm_max: int= 10, known_districts:bool=False, buffer: float = 0.0005, districts_of_interest: list =[1, 2, 3, 4]):

    gdf = gdf_input.copy()
    gdf, x_min, y_min, x_max, y_max = filter_for_geographic_section(gdf)
    # gdf = gdf[gdf["og_highway"].isin([1,])]

    fig, ax = plt.subplots(1, 1, figsize=(15, 15))    
    
    norm = get_norm(column_to_plot=column_to_plot, use_fixed_norm=use_fixed_norm, fixed_norm_max=fixed_norm_max, gdf=gdf)
    relevant_area_to_plot = get_relevant_area_to_plot(alpha, known_districts, buffer, districts_of_interest, gdf, ax, column_to_plot, norm, "og_vol_base_case", "og_highway")
    relevant_area_to_plot.plot(ax=ax, edgecolor='black', linewidth=2, facecolor='None', zorder=2)

    cbar = plotting(font, x_min, y_min, x_max, y_max, fig, ax, norm)
    
    cbar.set_label('Car volume: Difference to base case (%)', fontname=font, fontsize=15)
    if save_it:
        p = "predicted" if is_predicted else "actual"
        identifier = "n_" + str(number_to_plot) if number_to_plot is not None else zone_to_plot
        plt.savefig("results/" + identifier + "_" + p, bbox_inches='tight')
    plt.show()
    
def get_relevant_area_to_plot(alpha, known_districts, buffer, districts_of_interest, gdf, ax, column_to_plot, norm, volume_column, highway_column):
    gdf['og_capacity_reduction_rounded'] = gdf['og_capacity_reduction'].round(decimals=3)
    tolerance = 1e-3
    edges_with_capacity_reduction = gdf[np.abs(gdf['og_capacity_reduction_rounded']) > tolerance]

    # Normalize the values in the volume_column to a range suitable for linewidths
    # min_linewidth = 1
    
    # max_linewidth = 10
    # norm_volumes = (gdf[volume_column] - gdf[volume_column].min()) / (gdf[volume_column].max() - gdf[volume_column].min())
    # linewidths = min_linewidth + norm_volumes * (max_linewidth - min_linewidth)
    
    # Define the fixed linewidths based on the values in the highway_column
    def get_linewidth(value):
        if value in [0, 1]:
            return 5
        elif value == 2:
            return 3
        elif value == 3:
            return 2
        else:
            return 1

    # Apply the linewidth mapping
    linewidths = gdf[highway_column].apply(get_linewidth)
    gdf['linewidth'] = linewidths
    # Separate the GeoDataFrame into two groups based on linewidth
    large_lines = gdf[gdf['linewidth'] > 1]
    small_lines = gdf[gdf['linewidth'] == 1]

    
    # Create masks
    # mask_with_reduction = gdf.index.isin(edges_with_capacity_reduction.index)
    # mask_without_reduction = ~mask_with_reduction
    
    # gdf.plot(column=column_to_plot, cmap='coolwarm', linewidth=linewidths, ax=ax, legend=False,
    #                               norm=norm, label="Road network", zorder=2)
    
     # Plot small lines first
    small_lines.plot(column=column_to_plot, cmap='coolwarm', linewidth=small_lines['linewidth'], ax=ax, legend=False,
                    norm=norm, label="Street network", zorder=1)
    
    # Plot large lines after
    large_lines.plot(column=column_to_plot, cmap='coolwarm', linewidth=large_lines['linewidth'], ax=ax, legend=False,
                    norm=norm, label="Street network", zorder=2)
    
    
    coords = [(x, y) for geom in edges_with_capacity_reduction.geometry for x, y in zip(geom.xy[0], geom.xy[1])]
    alpha_shape = alphashape.alphashape(coords, alpha)
    relevant_area_to_plot = gpd.GeoSeries([alpha_shape], crs=gdf.crs)
    return relevant_area_to_plot

def get_norm(column_to_plot, use_fixed_norm, fixed_norm_max, gdf):
    if use_fixed_norm:
        norm = TwoSlopeNorm(vmin=-fixed_norm_max, vcenter=0, vmax=fixed_norm_max)
    else:
        print(gdf[column_to_plot].min())
        print(gdf[column_to_plot].median())
        print(gdf[column_to_plot].max())
        norm = TwoSlopeNorm(vmin=gdf[column_to_plot].min(), vcenter=gdf[column_to_plot].median(), vmax=gdf[column_to_plot].max())
    return norm
    
def filter_for_geographic_section(gdf):
    x_min = gdf.total_bounds[0] + 0.05
    y_min = gdf.total_bounds[1] + 0.05
    x_max = gdf.total_bounds[2]
    y_max = gdf.total_bounds[3]
    bbox = box(x_min, y_min, x_max, y_max)

    # Filter the network to include only the data within the bounding box
    gdf = gdf[gdf.intersects(bbox)]
    return gdf,x_min,y_min,x_max,y_max

def plotting(font, x_min, y_min, x_max, y_max, fig, ax, norm):
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)
    plt.xlabel("Longitude", fontname=font, fontsize=15)
    plt.ylabel("Latitude", fontname=font, fontsize=15)

    # Customize tick labels
    ax.tick_params(axis='both', which='major', labelsize=10)
    for label in (ax.get_xticklabels() + ax.get_yticklabels()):
        label.set_fontname(font)
        label.set_fontsize(15)
    
    # Create custom legend
    custom_lines = [Line2D([0], [0], color='grey', lw=4, label='Street network'),# Add more lines for other labels as needed
                    Line2D([0], [0], color='black', lw=2, label='Capacity was decreased in this section')]

    ax.legend(handles=custom_lines, prop={'family': font, 'size': 15})
    ax.set_position([0.1, 0.1, 0.75, 0.75])
    cax = fig.add_axes([0.87, 0.22, 0.03, 0.5])  # Manually position the color bar
    
    # Create the color bar
    sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=norm)
    sm._A = []
    cbar = plt.colorbar(sm, cax=cax)

    # Set color bar font properties
    cbar.ax.tick_params(labelsize=15)
    for t in cbar.ax.get_yticklabels():
        t.set_fontname(font)
    cbar.ax.yaxis.label.set_fontname(font)
    cbar.ax.yaxis.label.set_size(15)
    return cbar


# plot_combined_output(gdf_input=gdf_with_og_values, column_to_plot="og_vol_car_change_actual", save_it=True, 
#                         number_to_plot=None, zone_to_plot = "this",is_predicted=False,alpha=10,use_fixed_norm=True, 
#                         fixed_norm_max = 3,
#                         known_districts = True, buffer = 0.0005, districts_of_interest=districts_of_interest)
# hf.plot_difference_output(gdf_input=gdf_with_og_values, column1="og_vol_car_change_predicted", 
#                           column2="og_vol_car_change_actual", save_it=True, number_to_plot=None, zone_to_plot = zone_to_plot,
#                         alpha=10,use_fixed_norm=False,
#                         fixed_norm_max = 6,
#                         known_districts = True, buffer = 0.0005, districts_of_interest=districts_of_interest)

def compute_r2_torch(preds, targets):
    """Compute R^2 score using PyTorch."""
    mean_targets = torch.mean(targets)
    ss_tot = torch.sum((targets - mean_targets) ** 2)
    ss_res = torch.sum((targets - preds) ** 2)
    r2 = 1 - ss_res / ss_tot
    return r2

def filter_edge_index(edge_index, node_mask):
    # Create a map from old node indices to new node indices
    node_map = {old_idx.item(): new_idx for new_idx, old_idx in enumerate(node_mask.nonzero(as_tuple=True)[0])}
    
    # Filter edges
    mask = node_mask[edge_index[0]] & node_mask[edge_index[1]]
    filtered_edge_index = edge_index[:, mask]
    
    # Reindex edges
    filtered_edge_index = torch.tensor(
        [[node_map[old_idx.item()] for old_idx in filtered_edge_index[0]],
         [node_map[old_idx.item()] for old_idx in filtered_edge_index[1]]],
        dtype=torch.long
    )
    return filtered_edge_index

def validate_trained_model(model, valid_dl, loss_func, device):
    model.eval()
    val_loss = 0
    baseline_loss = 0
    num_batches = 0
    actual_vals = []
    predictions = []
    with torch.inference_mode():
        for idx, data in enumerate(valid_dl):
            input_node_features, targets = data.x.to(device), data.y.to(device)
            predicted = model(data.to(device))
            actual_vals.append(targets)
            predictions.append(predicted)

            val_loss += loss_func(predicted, targets).item()
            mean_t = torch.mean(targets)
            baseline_loss += loss_func(targets, torch.full_like(targets, mean_t))
            num_batches += 1
            
    actual_vals_cat = torch.cat(actual_vals)
    predictions_cat = torch.cat(predictions)
    r_squared = compute_r2_torch(preds=predictions_cat, targets=actual_vals_cat)
    return val_loss / num_batches, r_squared, actual_vals, predictions, baseline_loss/num_batches


# loss_fct = torch.nn.MSELoss()
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model = model.to(device)
# counter = 0

# i = 59

# test_loss, r_squared, actual_vals, predictions, baseline_loss = validate_one_model(model, test_set_loader.dataset[i], loss_fct, device)
# # pred_vs_baseline = test_loss/baseline_loss

# gdf = hf.data_to_geodataframe(data=test_set_loader.dataset[i], original_gdf=test_data, predicted_values=predictions)
# gdf_with_og_values = hf.map_to_original_values(input_gdf=gdf, scaler_x=scaler_x)
# gdf_with_og_values.crs = "EPSG:2154"
# gdf_with_og_values.to_crs("EPSG:4326", inplace=True)

# plot_combined_output(gdf_input=gdf_with_og_values, column_to_plot="og_vol_car_change_predicted", font='Times New Roman', 
#                     save_it=True, number_to_plot=i, zone_to_plot= "this_zone",
#                     is_predicted= True, alpha=100, 
#                     use_fixed_norm=True, 
#                     fixed_norm_max= 5,
#                     known_districts=False, 
#                     buffer = 0.0005, 
#                     districts_of_interest =[1, 2, 3, 4])

# plot_combined_output(gdf_input=gdf_with_og_values, column_to_plot="og_vol_car_change_actual", font='Times New Roman', 
#                     save_it=True, number_to_plot=i, zone_to_plot= "this_zone",
#                     is_predicted= False, alpha=100, 
#                     use_fixed_norm=True, 
#                     fixed_norm_max= 5,
#                     known_districts=False, 
#                     buffer = 0.0005, 
#                     districts_of_interest =[1, 2, 3, 4])

# # print(counter)

# def plot_combined_output(gdf_input: gpd.GeoDataFrame, column_to_plot: str, font: str = 'Times New Roman', 
#                          save_it: bool = False, number_to_plot: int = 0,
#                          zone_to_plot:str= "this_zone",
#                          is_predicted: bool = False, alpha:int=100, 
#                          use_fixed_norm:bool=True, 
#                          fixed_norm_max: int= 10, known_districts:bool=False, buffer: float = 0.0005, districts_of_interest: list =[1, 2, 3, 4]):

#     gdf = gdf_input.copy()
#     gdf, x_min, y_min, x_max, y_max = filter_for_geographic_section(gdf)
#     # gdf = gdf[gdf["og_highway"].isin([1,])]

#     fig, ax = plt.subplots(1, 1, figsize=(15, 15))    
    
#     norm = get_norm(column_to_plot=column_to_plot, use_fixed_norm=use_fixed_norm, fixed_norm_max=fixed_norm_max, gdf=gdf)
#     relevant_area_to_plot = get_relevant_area_to_plot(alpha, known_districts, buffer, districts_of_interest, gdf, ax, column_to_plot, norm, "og_vol_base_case", "og_highway")
#     relevant_area_to_plot.plot(ax=ax, edgecolor='black', linewidth=2, facecolor='None', zorder=2)

#     cbar = plotting(font, x_min, y_min, x_max, y_max, fig, ax, norm)
    
#     cbar.set_label('Car volume: Difference to base case (%)', fontname=font, fontsize=15)
#     if save_it:
#         p = "predicted" if is_predicted else "actual"
#         identifier = "n_" + str(number_to_plot) if number_to_plot is not None else zone_to_plot
#         plt.savefig("results/" + identifier + "_" + p, bbox_inches='tight')
#     plt.show()

In [2]:
# Parameters to define
run_path = '/home/enatterer/Development/gnn_predicting_effects_of_traffic_policies/data/runs_optimized/pnc_local_[256]_pnc_global_[512_256]_hidden_layer_str_[512_512_256_128_64]_dropout_0.3_use_dropout_False/'
point_net_conv_layer_structure_local_mlp = [256]
point_net_conv_layer_structure_global_mlp = [512,256]
gat_conv_layer_structure = [512,512,256,128,64]
dropout = 0.3
use_dropout = False 
in_channels = 6 
out_channels = 1 

model_path = run_path +  'trained_model/model.pth'
data_created_during_training = run_path + 'data_created_during_training/'
indices_of_datasets_to_use = [0, 1, 3, 4]

scaler_x = joblib.load(data_created_during_training + 'x_scaler.pkl')
scaler_pos = joblib.load(data_created_during_training + 'pos_scaler.pkl')

# Initialize the model
model = garch.MyGnn(in_channels=in_channels, out_channels=out_channels, 
                    point_net_conv_layer_structure_local_mlp=point_net_conv_layer_structure_local_mlp, 
                    point_net_conv_layer_structure_global_mlp = point_net_conv_layer_structure_global_mlp,
                    gat_conv_layer_structure=gat_conv_layer_structure,
                    dropout=dropout,
                    use_dropout=use_dropout)

# Load the model state dictionary
model.load_state_dict(torch.load(model_path))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

Initializing PointNetConv(local_nn=Sequential(
  (0): Linear(in_features=6, out_features=256, bias=True)
  (1): ReLU()
), global_nn=Sequential(
  (0): Linear(in_features=256, out_features=512, bias=True)
  (1): Linear(in_features=512, out_features=256, bias=True)
  (2): ReLU()
  (3): Linear(in_features=256, out_features=512, bias=True)
  (4): ReLU()
))
Initializing 0.weight with kaiming_normal
Initializing 0.bias with zeros
Initializing 0.weight with kaiming_normal
Initializing 0.bias with zeros
Initializing 1.weight with kaiming_normal
Initializing 1.bias with zeros
Initializing 3.weight with kaiming_normal
Initializing 3.bias with zeros
Initializing Linear(in_features=6, out_features=256, bias=True)
Initializing Linear(in_features=256, out_features=512, bias=True)
Initializing Linear(in_features=512, out_features=256, bias=True)
Initializing Linear(in_features=256, out_features=512, bias=True)
Initializing GATConv(512, 512, heads=1)
Initializing GATConv(512, 256, heads=1)
Initializin

In [3]:
# Load the test dataset created during training
test_set_dl = torch.load(data_created_during_training + 'test_dl.pt')

# Load the DataLoader parameters
with open(data_created_during_training + 'test_loader_params.json', 'r') as f:
    test_set_dl_loader_params = json.load(f)
    
# Remove or correct collate_fn if it is incorrectly specified
if 'collate_fn' in test_set_dl_loader_params and isinstance(test_set_dl_loader_params['collate_fn'], str):
    del test_set_dl_loader_params['collate_fn']  # Remove it to use the default collate function
    
test_set_loader = torch.utils.data.DataLoader(test_set_dl, **test_set_dl_loader_params)

In [4]:
test_data = "../../data/test_data/gdf_pop_1pm_policy_in_1_2_3_4.geojson"
test_data = gpd.read_file(test_data)
base_case = "../../data/test_data/gdf_basecase_mean_pop_1pm.geojson"
base_case = gpd.read_file(base_case)

In [5]:
i = 59
loss_fct= torch.nn.MSELoss()
test_loss, r_squared, actual_vals, predictions, baseline_loss = validate_one_model(model, test_set_loader.dataset[i], loss_fct, device)

gdf = hf.data_to_geodataframe(data=test_set_loader.dataset[0], original_gdf=test_data, predicted_values=predictions)
gdf_with_og_values = hf.map_to_original_values(input_gdf=gdf, scaler_x=scaler_x)
gdf_with_og_values.crs = "EPSG:2154"
gdf_with_og_values.to_crs("EPSG:4326", inplace=True)

In [6]:
indices_roads_with_highway_primary_sec_tertiary = gdf_with_og_values[gdf_with_og_values['og_highway'].isin([1,2,3])].index
indices_roads_with_highway_primary_ = gdf_with_og_values[gdf_with_og_values['og_highway'].isin([1])].index
indices_roads_with_highway_sec = gdf_with_og_values[gdf_with_og_values['og_highway'].isin([2])].index
indices_roads_with_highway_tertiary = gdf_with_og_values[gdf_with_og_values['og_highway'].isin([3])].index

indices_roads_with_highway_not_primary_sec_tertiary = gdf_with_og_values[~gdf_with_og_values['og_highway'].isin([1, 2, 3])].index

gdf_with_og_values['og_capacity_reduction_rounded'] = gdf_with_og_values['og_capacity_reduction'].round(decimals=3)
tolerance = 1e-3
indices_roads_with_cap_reduction = gdf_with_og_values[gdf_with_og_values['og_capacity_reduction_rounded'] < -1e-3].index
indices_roads_with_no_cap_reduction = gdf_with_og_values[gdf_with_og_values['og_capacity_reduction_rounded'] >= -1e-3].index

indices_roads_with_highway_primary_sec_tertiary_and_cap_reduction = gdf_with_og_values[
    (gdf_with_og_values['og_highway'].isin([1, 2, 3])) & 
    (gdf_with_og_values['og_capacity_reduction_rounded'] < -1e-3)
].index
indices_roads_with_highway_primary_sec_tertiary_and_not_cap_reduction = gdf_with_og_values[
    (gdf_with_og_values['og_highway'].isin([1, 2, 3])) & 
    (gdf_with_og_values['og_capacity_reduction_rounded'] >= -1e-3)
].index

no_filter = gdf_with_og_values.index

In [7]:
districts_of_interest = [1, 2, 3, 4, 6, 8, 9, 11, 14, 20]
districts = gpd.read_file("../../data/visualisation/districts_paris.geojson")
target_districts = districts[districts['c_ar'].isin(districts_of_interest)]
gdf_with_og_values['intersects_target_districts'] = gdf_with_og_values.apply(lambda row: target_districts.intersects(row.geometry).any(), axis=1)

In [8]:
indices_this_zone = gdf_with_og_values[gdf_with_og_values['intersects_target_districts']].index
overlap = indices_this_zone.intersection(indices_roads_with_highway_primary_sec_tertiary)

indices_to_filter_for = overlap

filtered_actual = actual_vals[indices_to_filter_for]
filtered_actual_mean = torch.mean(filtered_actual)
filtered_predicted = predictions[indices_to_filter_for]

mse_filtered = loss_fct(filtered_actual, filtered_predicted)
baseline_filtered = loss_fct(filtered_actual, torch.full_like(filtered_actual, filtered_actual_mean))
loss_fct_aux = torch.nn.MSELoss(reduction='none')
variance = torch.var(loss_fct_aux(filtered_actual, torch.full_like(filtered_actual, filtered_actual_mean)))
r_squared = compute_r2_torch(preds=filtered_predicted, targets=filtered_actual)
print(baseline_filtered)
print(mse_filtered)
print(r_squared)
loss_fct_aux = torch.nn.MSELoss(reduction='none')
variance = torch.var(loss_fct_aux(filtered_actual, torch.full_like(filtered_actual, filtered_actual_mean)))
print(f'variance: {variance}')


# actual_mean = torch.mean(actual_vals)
# variance = torch.var(loss_fct_aux(actual_vals, torch.full_like(actual_vals, actual_mean)))
# print(f'Test Loss: {test_loss}')
# print(f'Baseline Loss: {baseline_loss}')
# print(f'r_squared: {r_squared}')
# print(f'variance: {variance}')
# # print("Number of values where 'og_capacity_reduction' is not 0 with tolerance 1e-3:", num_values_not_zero)

# gdf_in_meters = gdf_with_og_values.to_crs("EPSG:32633")
# gdf_in_meters.length

# tolerance = 1e-3
# gdf_with_capacity_reduction = gdf_in_meters[abs(gdf_in_meters['og_capacity_reduction']) > tolerance]
# gdf_with_capacity_reduction['length'] = gdf_with_capacity_reduction.length
# total_length = gdf_with_capacity_reduction['length'].sum()/1000

tensor(12.1289, device='cuda:0')
tensor(5.4791, device='cuda:0')
tensor(0.5483, device='cuda:0')
variance: 785.4876098632812


In [9]:
# loss_fct = torch.nn.MSELoss()

# test_loss, r_squared, actual_vals, predictions, baseline_loss = validate_trained_model(model, test_set_loader.dataset, loss_fct, device)
# print(f'Test Loss: {test_loss}')
# print(f'r_squared: {r_squared}')
# print(f'baseline_loss: {baseline_loss}')

# print(f'actual_vals shape: {len(actual_vals)}')
# print(f'predictions shape: {len(predictions)}')



In [10]:
# Function to filter list of tensors
# def filter_tensors_by_indices(tensor_list, indices):
#     return [tensor[indices] for tensor in tensor_list]

In [11]:
# Ensure actual_vals and predictions are tensors
# if isinstance(actual_vals, list):
#     actual_vals = torch.tensor(actual_vals)
# if isinstance(predictions, list):
#     predictions = torch.tensor(predictions)
# THIS IS FOR WHOLE TEST SET ! 


# # Filter actual and predicted values
# filtered_actual = filter_tensors_by_indices(actual_vals, indices_to_filter_for)
# filtered_predicted = filter_tensors_by_indices(predictions, indices_to_filter_for)

# # Concatenate filtered tensors for computing metrics
# filtered_actual_concat = torch.cat(filtered_actual)
# filtered_predicted_concat = torch.cat(filtered_predicted)

# filtered_actual_mean = torch.mean(filtered_actual_concat)
# # filtered_actual = actual_vals[indices_to_filter_for]
# # filtered_actual_mean = torch.mean(filtered_actual)
# # filtered_predicted = predictions[indices_to_filter_for]

# mse_filtered = loss_fct(filtered_actual_concat, filtered_predicted_concat)
# baseline_filtered = loss_fct(filtered_actual_concat, torch.full_like(filtered_actual_concat, filtered_actual_mean))
# r_squared = compute_r2_torch(preds=filtered_predicted_concat, targets=filtered_actual_concat)
# loss_fct_aux = torch.nn.MSELoss(reduction='none')
# variance = torch.var(loss_fct_aux(filtered_actual_concat, torch.full_like(filtered_actual_concat, filtered_actual_mean)))

# print(f'variance: {variance}')
# print(f'Baseline Loss: {baseline_filtered}')
# print(f'Test Loss: {mse_filtered}')
# print(f'r_squared: {r_squared}')


In [12]:
indices_to_filter_for = no_filter

In [13]:
# # gdf = hf.data_to_geodataframe(data=test_set_loader.dataset[0], original_gdf=test_data, predicted_values=predictions)
# # gdf_with_og_values = hf.map_to_original_values(input_gdf=gdf, scaler_x=scaler_x)
# # gdf_with_og_values.crs = "EPSG:2154"
# # gdf_with_og_values.to_crs("EPSG:4326", inplace=True)

# gdf_in_meters = gdf_with_og_values.to_crs("EPSG:32633")
# gdf_in_meters.length

# tolerance = 1e-3
# gdf_with_capacity_reduction = gdf_in_meters.loc[indices_to_filter_for]
# gdf_with_capacity_reduction['length'] = gdf_with_capacity_reduction.length
# total_length = gdf_with_capacity_reduction['length'].sum()/1000
# print(total_length)

In [14]:
# total_length = gdf_with_capacity_reduction['length'].sum()

In [15]:
gdf_in_meters = gdf_with_og_values.to_crs("EPSG:32633")
gdf_in_meters.length

tolerance = 1e-3
gdf_with_capacity_reduction = gdf_in_meters[abs(gdf_in_meters['og_capacity_reduction']) > tolerance]
gdf_with_capacity_reduction['length'] = gdf_with_capacity_reduction.length
total_length = gdf_with_capacity_reduction['length'].sum()/1000

/opt/anaconda3/envs/chenhao-gnn/lib/python3.10/site-packages/geopandas/geodataframe.py:1525: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [16]:
len(gdf_with_capacity_reduction)

4492